# Gold model and business reports
Full overwrite. Chạy sau Silver thành công. Tập chính theo đề, thêm báo cáo Completed để thuyết minh.

In [ ]:
run_id = 'manual'
source_folder = 'Source'
bronze_folder = 'Bronze'
reports_folder = 'Reports'
table_schema = 'dbo'
strict_project_snapshot = True

In [ ]:
from pyspark.sql import functions as F, types as T
spark.conf.set("spark.sql.session.timeZone", "UTC")
def save(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
def key(*cols):
    return F.sha2(F.to_json(F.struct(*[F.col(c) for c in cols])), 256)

In [ ]:
import re
from pyspark.sql import functions as F, types as T
assert re.fullmatch(r'[A-Za-z0-9_-]+', run_id), 'Invalid run_id'
assert not table_schema or re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',table_schema)
def table_name(name):
    assert re.fullmatch(r'[a-z_]+',name)
    return f'{table_schema}.{name}' if table_schema else name
def table(name): return spark.table(table_name(name))
def save(df, name):
    df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(table_name(name))
def bronze_path(kind): return f'Files/{bronze_folder}/{run_id}/{kind}/'
def source_frame(filename):
    return (spark.read.option('header',True).option('inferSchema',False).option('quote','"')
        .option('escape','"').option('mode','FAILFAST').csv(f'Files/{source_folder}/{filename}'))

In [ ]:
o = table('silver_sales')
i = table('silver_sales_items')
r = table('silver_exchange_rate')
o = o.withColumn('year',F.year('order_date')).withColumn('month',F.month('order_date'))
rates = r.select('year','month',F.col('from_currency').alias('currency'),'exchange_rate')
assert rates.groupBy('year','month','currency').count().filter('count > 1').count()==0
o = o.join(rates,['year','month','currency'],'left')
assert o.filter('exchange_rate is null').count()==0, 'Missing exchange rate: stop publication'
o = (o.withColumn('date_key',F.date_format('order_date','yyyyMMdd').cast('int'))
    .withColumn('customer_key',key('customer_name','customer_email','customer_phone'))
    .withColumn('location_key',key('location')).withColumn('payment_key',key('payment_method'))
    .withColumn('has_discount',F.coalesce(F.length(F.trim('discount_code'))>0,F.lit(False))))
dates = spark.sql("SELECT explode(sequence(to_date('2024-01-01'),to_date('2025-12-31'),interval 1 day)) AS full_date")
dates = dates.select(F.date_format('full_date','yyyyMMdd').cast('int').alias('date_key'),'full_date',F.year('full_date').alias('year'),F.month('full_date').alias('month'))
dimensions = {
 'gold_dim_date':dates,
 'gold_dim_customer':o.select('customer_key','customer_name','customer_email','customer_phone').distinct(),
 'gold_dim_location':o.select('location_key','location').distinct(),
 'gold_dim_payment_method':o.select('payment_key','payment_method').distinct(),
 'gold_dim_product':i.withColumn('product_key',key('product_id','category')).select('product_key','product_id','category').distinct()}
for name,df in dimensions.items():
    pk = df.columns[0]
    assert df.select(pk).distinct().count()==df.count(), name+' duplicate key'
fact_orders = o.select('order_id','date_key','customer_key','location_key','payment_key','customer_age','device_type','currency',
    'total_amount_usd','shipping_cost_usd','exchange_rate','order_status','feedback_score','loyalty_points','report_eligible','has_discount','discount_code')
assert i.join(o.select('order_id'),'order_id','left_anti').count()==0
from pyspark.sql.window import Window
w = Window.partitionBy('order_id')
lines = i.withColumn('product_key',key('product_id','category')).withColumn('line_subtotal_usd',(F.col('price')*F.col('quantity')).cast('decimal(20,2)'))
lines = lines.withColumn('order_subtotal_usd',F.sum('line_subtotal_usd').over(w)).join(o.select('order_id','date_key','total_amount_usd','exchange_rate'),'order_id')
assert lines.filter('order_subtotal_usd <= 0').count()==0
# Round to cents, then assign rounding residual to the last source item.
lines = lines.withColumn('rounded_allocation',(F.col('total_amount_usd')*F.col('line_subtotal_usd')/F.col('order_subtotal_usd')).cast('decimal(20,2)'))
lines = lines.withColumn('allocated_revenue_usd',(
    F.col('rounded_allocation') + F.when(F.col('item_index')==F.max('item_index').over(w),
    F.col('total_amount_usd')-F.sum('rounded_allocation').over(w)).otherwise(F.lit(0))).cast('decimal(20,2)'))
fact_items = lines.select('order_id','item_index','date_key','product_key','price','quantity','line_subtotal_usd','allocated_revenue_usd',
    (F.col('allocated_revenue_usd')*F.col('exchange_rate')).cast('decimal(30,6)').alias('allocated_revenue_vnd'))
assert fact_items.groupBy('order_id','item_index').count().filter('count>1').count()==0
totals = fact_items.groupBy('order_id').agg(F.sum('allocated_revenue_usd').alias('allocated')).join(o.select('order_id','total_amount_usd'),'order_id')
assert totals.filter('allocated != total_amount_usd').count()==0
assert totals.count()==o.count()
for fk, dim in [('date_key','gold_dim_date'),('customer_key','gold_dim_customer'),('location_key','gold_dim_location'),('payment_key','gold_dim_payment_method')]:
    assert fact_orders.join(dimensions[dim].select(fk),fk,'left_anti').count()==0, 'Missing foreign key: '+fk
assert fact_items.join(dimensions['gold_dim_product'].select('product_key'),'product_key','left_anti').count()==0
for name,df in dimensions.items(): save(df,name)
save(fact_orders,'gold_fact_orders');save(fact_items,'gold_fact_order_items')
print('Dimensions and facts validated and saved')

In [ ]:
eligible = fact_orders.filter('report_eligible')
assert eligible.filter("order_status='Failed' OR feedback_score IS NULL OR feedback_score NOT BETWEEN 1 AND 5").count()==0
detail = fact_items.join(eligible.select('order_id','order_status'),'order_id').join(dimensions['gold_dim_product'],'product_key').join(dates,'date_key')
def revenue(df):
    return df.groupBy('year','month','category').agg(F.sum('allocated_revenue_vnd').alias('total_revenue_vnd'))
monthly = revenue(detail)
completed = revenue(detail.filter("order_status='Completed'"))
promo = eligible.join(dimensions['gold_dim_location'],'location_key').groupBy('location').agg(
    F.count('*').alias('total_orders'),F.sum(F.col('has_discount').cast('long')).alias('orders_with_discount'))
promo = promo.withColumn('discount_order_pct',(100*F.col('orders_with_discount')/F.col('total_orders')).cast('decimal(7,2)'))
assert promo.filter('discount_order_pct < 0 OR discount_order_pct > 100').count()==0
assert promo.agg(F.sum('total_orders')).first()[0]==eligible.count()
expected = eligible.select((F.col('total_amount_usd')*F.col('exchange_rate')).cast('decimal(30,6)').alias('v')).agg(F.sum('v')).first()[0]
actual = monthly.agg(F.sum('total_revenue_vnd')).first()[0]
assert expected == actual, f'Revenue mismatch: {expected} != {actual}'
save(monthly,'gold_monthly_revenue_category')
save(completed,'gold_monthly_completed_revenue_category')
save(promo,'gold_promotion_by_location')
display(monthly.orderBy('year','month','category'))
display(completed.orderBy('year','month','category'))
display(promo.orderBy(F.desc('total_orders')))
print({'eligible_orders':eligible.count(),'completed_orders':eligible.filter("order_status='Completed'").count(),'total_revenue_vnd':str(actual)})